<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/dl-fundamentals/debug-training/debug-training.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("debug-training", ...)`


# Debug a Broken Training Pipeline

**Difficulty**: 🟡 Medium

### Problem Statement

*"The model trains, but something is off."* Research-engineer and research-scientist interviews often hand you a hundred lines of training code and ask you to find what is wrong. Nothing crashes. The bugs are **silent** and show up only as symptoms: a loss that stalls at a suspicious value, an accuracy that changes when you evaluate the same model twice, memory that grows every epoch, a model that trains fine yet scores 0%. What is being tested is diagnostic reasoning, going from **symptom → hypothesis → targeted test → fix**. Recall of APIs matters much less.

Below is a complete, small image-classification pipeline on a synthetic dataset: 16×16 RGB "photos" of four procedurally drawn shapes, stored as `uint8` in HWC layout like images decoded by PIL/OpenCV. Nothing is downloaded. The pipeline runs end to end without errors and contains **exactly 9 bugs** spread over its six functions. They are not spread evenly: a function may have several bugs, or none.

---

### Requirements

Fix the six functions so each does what its docstring says (keep the signatures):

1. **`to_tensor(images)`**: `uint8 (N, H, W, C)` → `float32 (N, C, H, W)` in $[0, 1]$, with `out[n, c, h, w] = images[n, h, w, c] / 255`.
2. **`compute_normalization_stats(images)`**: float `(N, C, H, W)` → `(mean, std)`, each of shape `(C,)`, i.e. statistics of each channel over all of $N$, $H$, $W$.
3. **`normalize(images, mean, std)`**: per-channel $(x - \mu_c)/\sigma_c$.
4. **`make_model(num_classes)`**: a CNN mapping `(B, 3, 16, 16)` → `(B, num_classes)` **logits** for `nn.CrossEntropyLoss`.
5. **`train_one_epoch(model, loader, optimizer, loss_fn) -> float`**: one pass over `loader` with a standard training step per batch. The update for a batch must depend only on that batch, and dropout/BatchNorm must behave as in training. Returns the mean of the per-batch losses as a Python `float`.
6. **`evaluate(model, loader) -> float`**: accuracy in $[0, 1]$ over all samples of `loader`, with dropout/BatchNorm in their inference behaviour and no memory wasted on things inference does not need. Evaluating the same model twice must give the same number, and evaluation must not change the model.

Once fixed, the driver cell reaches **> 95% test accuracy in 5 epochs** on CPU in a few seconds.

---

### Constraints

- ✅ Fix the bugs in place with minimal diffs, as you would in a code review, and note for each one the **symptom** it causes.
- ❌ Do not change the dataset generator, the driver cell, the optimizer or the hyper-parameters.
- ✅ Try to find all 9 from the symptoms and by reading the code **before** running `check(...)`. You won't have a grader in an interview. The grader tests each function's behaviour separately, so it then points at whatever is left.

---

<details>
  <summary>💡 Hint</summary>

  - **Look at the data after every transform.** The ASCII viewer below prints an image. Does it still look like the shape it is labelled as?
  - After normalisation, each channel of the *training* set should have mean ≈ 0 and std ≈ 1. Print it.
  - **Know your reference numbers.** The initial loss should be ≈ $\log C$. Is there a floor the loss cannot go below? What would cause one?
  - **Evaluate the same model twice.** Different answers mean something random is still switched on.
  - Check the *type* of what the training loop returns. Does it have a `.grad_fn`?
  - For each line of each loop, ask: what state does this depend on (`.grad`, `model.training`, the autograd graph), and who resets it?

</details>

---

In [ ]:
import math

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
NUM_CLASSES = 4
CLASS_NAMES = ["horizontal bar", "vertical bar", "hollow square", "diagonal"]


def make_shapes_dataset(n, seed, size=16):
    # Synthetic "photos": uint8 (n, size, size, 3) in HWC layout, like decoded JPEGs. Labels (n,).
    # This generator is correct -- the bugs are elsewhere.
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, NUM_CLASSES, (n,), generator=g)
    background = torch.tensor([0.15, 0.45, 0.70])                 # each channel has its own level
    imgs = background + 0.08 * torch.randn(n, size, size, 3, generator=g)
    yy = torch.arange(size).view(1, size, 1)
    xx = torch.arange(size).view(1, 1, size)
    r = torch.randint(4, size - 4, (n, 1, 1), generator=g)        # random shape centre
    c = torch.randint(4, size - 4, (n, 1, 1), generator=g)
    dy, dx = yy - r, xx - c
    shapes = torch.stack([
        (dy.abs() <= 1) & (dx.abs() <= 4),                                          # horizontal bar
        (dx.abs() <= 1) & (dy.abs() <= 4),                                          # vertical bar
        (dy.abs() <= 3) & (dx.abs() <= 3) & ~((dy.abs() <= 1) & (dx.abs() <= 1)),   # hollow square
        ((dy - dx).abs() <= 1) & (dy.abs() <= 4),                                   # diagonal
    ], dim=1)                                                     # (n, 4, size, size)
    mask = shapes[torch.arange(n), labels]                        # (n, size, size)
    color = (1 - background) + 0.15 * torch.randn(n, 1, 1, 3, generator=g)
    imgs = torch.where(mask[..., None], color, imgs)
    return (imgs.clamp(0, 1) * 255).round().to(torch.uint8), labels


train_images, train_labels = make_shapes_dataset(1024, seed=0)
test_images, test_labels = make_shapes_dataset(512, seed=1)
print("train:", tuple(train_images.shape), train_images.dtype, " test:", tuple(test_images.shape))
print("class counts:", torch.bincount(train_labels).tolist())

In [ ]:
# Debugging toolkit (correct, use it freely)
def ascii_image(gray_hw):
    # Print a (H, W) image as ASCII art, min-max scaled.
    ramp = " .:-=+*#%@"
    g = gray_hw.float()
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    for row in g:
        print(" ".join(ramp[int(v * (len(ramp) - 1))] for v in row))


print(f"train_images[0] is a {CLASS_NAMES[train_labels[0]]}. Its red channel (raw HWC):")
ascii_image(train_images[0, :, :, 0])

### The pipeline (contains exactly 9 bugs)

In [ ]:
def to_tensor(images):
    """uint8 (N, H, W, C) images in HWC layout -> float32 (N, C, H, W) in [0, 1]."""
    n, h, w, c = images.shape
    return images.reshape(n, c, h, w).float() / 255.0


def compute_normalization_stats(images):
    """Per-channel mean and std of a float (N, C, H, W) batch -> two tensors of shape (C,)."""
    num_channels = images.shape[1]
    per_channel = images.reshape(num_channels, -1)            # one row per channel
    return per_channel.mean(dim=1), per_channel.std(dim=1)


def normalize(images, mean, std):
    """Standardise each channel: (x - mean[c]) / std[c]."""
    return (images - mean.view(1, -1, 1, 1)) / std.view(1, -1, 1, 1)

In [ ]:
def make_model(num_classes=NUM_CLASSES):
    """Small CNN: (B, 3, 16, 16) -> (B, num_classes) logits for nn.CrossEntropyLoss."""
    return nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
        nn.MaxPool2d(2),                                          # 16x16 -> 8x8
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
        nn.MaxPool2d(2),                                          # 8x8 -> 4x4
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(32 * 4 * 4, num_classes),
        nn.Softmax(dim=1),
    )

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    """One pass over `loader`, one optimizer step per batch. Returns the mean batch loss (float)."""
    total_loss = 0.0
    for images, labels in loader:
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss
    return total_loss / len(loader)


def evaluate(model, loader):
    """Accuracy in [0, 1] of `model` over every sample in `loader`."""
    correct, total = 0, 0
    for images, labels in loader:
        logits = model(images)
        preds = logits.max(dim=1)[0]
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / total

### Run it

In [ ]:
# The driver: do not modify. Run it, read the output, form hypotheses.
torch.manual_seed(0)
train_x, test_x = to_tensor(train_images), to_tensor(test_images)
print(f"after to_tensor: {tuple(train_x.shape)}. Red channel of the same {CLASS_NAMES[train_labels[0]]}:")
ascii_image(train_x[0, 0])

mean, std = compute_normalization_stats(train_x)
train_x, test_x = normalize(train_x, mean, std), normalize(test_x, mean, std)
print("normalised train set  per-channel mean:", [round(v, 3) for v in train_x.mean(dim=(0, 2, 3)).tolist()],
      " std:", [round(v, 3) for v in train_x.std(dim=(0, 2, 3)).tolist()])

train_loader = DataLoader(TensorDataset(train_x, train_labels), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(test_x, test_labels), batch_size=256)
model = make_model(NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()
print(f"reference: chance-level loss log(C) = {math.log(NUM_CLASSES):.3f}")

history = []
for epoch in range(5):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    acc = evaluate(model, test_loader)
    acc_again = evaluate(model, test_loader)            # same model, same data
    history.append(train_loss)
    print(f"epoch {epoch}: train loss {float(train_loss):.4f} [{type(train_loss).__name__}]  "
          f"test acc {acc:.3f}  (evaluated again: {acc_again:.3f})")

### Read the symptoms

Before touching the code, write down what the output above tells you. Questions worth asking:

1. Does the image after `to_tensor` still show the same shape as the raw image?
2. Are the normalised per-channel means ≈ 0 and stds ≈ 1?
3. Where does the training loss start, where does it go, and what type is it?
4. Why could the test accuracy be near 0%, which is *worse* than guessing, while the loss moves?
5. Why would two evaluations of the same model disagree?

Then read each function line by line and list the bugs, each with the symptom it causes. Fix them one at a time and re-run the driver. Some bugs only become visible once another one is fixed.

In [ ]:
# What a healthy pipeline looks like (these pass once all 9 bugs are fixed)
assert isinstance(history[-1], float), \
    f"train_one_epoch should return a plain float, got {type(history[-1]).__name__}"
assert acc == acc_again, "evaluating the same model twice must give the same accuracy"
norm_mean = train_x.mean(dim=(0, 2, 3))
assert torch.allclose(norm_mean, torch.zeros(3), atol=1e-3), \
    f"normalised training data should have per-channel mean 0, got {norm_mean.tolist()}"
assert history[-1] < 0.3, f"final train loss {history[-1]:.3f} -- this task is easy"
assert acc > 0.95, f"test accuracy {acc:.3f} -- a healthy pipeline gets > 95% here"
print("pipeline looks healthy")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("debug-training", to_tensor=to_tensor, compute_normalization_stats=compute_normalization_stats,
      normalize=normalize, make_model=make_model, train_one_epoch=train_one_epoch, evaluate=evaluate)